In [ ]:
# ── Cell 1: Install & import dependencies ────────────────────────────────────
import subprocess, sys

def _pip(pkg: str, import_as=None):
    try:
        __import__(import_as or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"  ✓ {pkg} installed.")

_pip("python-docx", "docx")
_pip("pandas")
_pip("openpyxl")

import os, glob, json as _json, re
from pathlib import Path
import docx
from docx.oxml.ns import qn
import pandas as pd
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment

print("All imports OK.")

In [ ]:
# ── Cell 2: Configuration ────────────────────────────────────────────────────
# ↓ Set this to the folder that contains your .docx MoM files
DOCX_FOLDER = Path(r"c:\Users\ben.lu\OneDrive - shl-group.com\Documents\Privacy\python2\Ben\SHL\Tool Assessment\MoM extraction\MoM")

# Output files — both are saved inside DOCX_FOLDER
PHASE1_XLSX = DOCX_FOLDER / "mom_phase1_preview.xlsx"   # Phase 1 raw extraction
OUTPUT_XLSX = DOCX_FOLDER / "mom_table.xlsx"             # Final summary table

print(f"DOCX folder     : {DOCX_FOLDER}")
print(f"Phase 1 preview : {PHASE1_XLSX}")
print(f"Final output    : {OUTPUT_XLSX}")
print(f"Files found     : {len(list(DOCX_FOLDER.glob('*.docx')))}")

In [ ]:
# ── Cell 3: Helper functions ─────────────────────────────────────────────────

_HL_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

_SECTION_MAP = [
    (["agenda"],                "Agenda"),
    (["root cause"],            "Root Cause"),
    (["solution", "solutions"], "Solutions"),
]

_HEADER_KEYS = {
    "subject":          "Subject",
    "no":               "No",
    "number":           "No",
    "date":             "Date",
    "place":            "Place",
    "location":         "Place",
    "recorded by":      "Recorded by",
    "recorder":         "Recorded by",
    "approved by":      "Approved by",
    "approver":         "Approved by",
    "issue type":       "Issue Type",
    "type":             "Issue Type",
    "counter previous": "Counter Previous",
    "counter pre":      "Counter Previous",
    "previous":         "Counter Previous",
    "counter current":  "Counter Current",
    "counter cur":      "Counter Current",
    "current":          "Counter Current",
}

def _strip_enum(text: str) -> str:
    return re.sub(r"^\s*[\d\w]+[\.\)]\s*", "", text).strip()

def _match_section(text: str):
    norm = _strip_enum(text).lower()
    for keywords, name in _SECTION_MAP:
        for kw in keywords:
            if norm == kw or norm.startswith(kw + " ") or norm.startswith(kw + ":"):
                return name
    return None

def _is_issue_heading(text: str) -> bool:
    return bool(re.match(r"^\s*\d+\.\s+\S", text))

def extract_raw(docx_path: Path) -> dict:
    doc = docx.Document(str(docx_path))
    part_rels = doc.part.rels

    def _para_links(para_elem):
        links = []
        for hl in para_elem.findall(".//" + qn("w:hyperlink")):
            display = "".join(t.text for t in hl.iter(qn("w:t"))).strip()
            r_id    = hl.get(f"{{{_HL_NS}}}id")
            href    = part_rels[r_id].target_ref if (r_id and r_id in part_rels) else ""
            links.append({"text": display, "href": href})
        return links

    body_elements  = []
    all_hyperlinks = []

    for child in doc.element.body.iterchildren():
        tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
        if tag == "p":
            text   = "".join(t.text for t in child.iter(qn("w:t"))).strip()
            pStyle = child.find(".//" + qn("w:pStyle"))
            style  = pStyle.get(qn("w:val")) if pStyle is not None else "Normal"
            links  = _para_links(child)
            all_hyperlinks.extend(links)
            body_elements.append({"kind": "paragraph", "text": text,
                                   "style": style, "hyperlinks": links})
        elif tag == "tbl":
            rows = [
                ["".join(t.text for t in c.iter(qn("w:t"))).strip()
                 for c in row_el.iter(qn("w:tc"))]
                for row_el in child.iter(qn("w:tr"))
            ]
            body_elements.append({"kind": "table", "data": rows})

    return {
        "body_elements": body_elements,
        "paragraphs":    [e for e in body_elements if e["kind"] == "paragraph"],
        "tables":        [e["data"] for e in body_elements if e["kind"] == "table"],
        "hyperlinks":    all_hyperlinks,
        "has_image":     len(doc.inline_shapes) > 0,
    }

def parse_header(raw: dict) -> dict:
    fields = list(dict.fromkeys(_HEADER_KEYS.values()))
    header = {f: "" for f in fields}
    if not raw["tables"]:
        return header
    flat = [cell for row in raw["tables"][0] for cell in row]
    for i, cell in enumerate(flat):
        norm = cell.strip().rstrip(":").lower()
        for pattern, field in _HEADER_KEYS.items():
            if norm == pattern:
                for j in range(i + 1, min(i + 4, len(flat))):
                    val = flat[j].strip()
                    if val and val.rstrip(":").lower() not in _HEADER_KEYS:
                        if not header[field]:
                            header[field] = val
                        break
                break
    return header

def _parse_sections(elements: list) -> dict:
    text_buckets  = {"Agenda": [], "Root Cause": [], "Solutions": []}
    table_buckets = {"Solutions": []}
    agenda_links  = []
    current_sec   = None
    for elem in elements:
        if elem["kind"] == "paragraph":
            text = elem["text"]
            if not text:
                continue
            sec = _match_section(text)
            if sec:
                current_sec = sec
                continue
            if current_sec:
                text_buckets[current_sec].append(text)
                if current_sec == "Agenda":
                    for hl in elem.get("hyperlinks", []):
                        if hl["href"]:
                            agenda_links.append(hl["href"])
        elif elem["kind"] == "table" and current_sec == "Solutions":
            table_buckets["Solutions"].append(elem["data"])
    solutions = "\n".join(text_buckets["Solutions"])
    for tbl in table_buckets["Solutions"]:
        solutions += "\n[TABLE]\n" + _json.dumps(tbl, ensure_ascii=False)
    return {
        "Agenda":      "\n".join(text_buckets["Agenda"]),
        "Agenda Link": ", ".join(agenda_links),
        "Root Cause":  "\n".join(text_buckets["Root Cause"]),
        "Solutions":   solutions,
    }

def parse_body(raw: dict) -> list:
    elements  = raw["body_elements"]
    first_tbl = next((i for i, e in enumerate(elements) if e["kind"] == "table"), -1)
    body      = elements[first_tbl + 1:] if first_tbl >= 0 else elements
    splits    = [i for i, e in enumerate(body)
                 if e["kind"] == "paragraph" and _is_issue_heading(e["text"])]
    if not splits:
        sections = _parse_sections(body)
        sections["issue_heading"] = ""
        return [sections]
    issues = []
    for k, start in enumerate(splits):
        end            = splits[k + 1] if k + 1 < len(splits) else len(body)
        sections       = _parse_sections(body[start + 1 : end])
        sections["issue_heading"] = body[start]["text"]
        issues.append(sections)
    return issues

COLUMNS = [
    "檔名", "Subject", "No", "Date", "Place",
    "Recorded by", "Approved by", "Issue Type",
    "Counter Previous", "Counter Current",
    "Agenda", "Agenda Link", "Root Cause", "Solutions", "Has Image",
]

def build_rows(filename: str, header: dict, issues: list, has_image: bool) -> list:
    rows = []
    for issue in issues:
        rows.append({
            "檔名":             filename,
            "Subject":          header.get("Subject",          ""),
            "No":               header.get("No",               ""),
            "Date":             header.get("Date",             ""),
            "Place":            header.get("Place",            ""),
            "Recorded by":      header.get("Recorded by",      ""),
            "Approved by":      header.get("Approved by",      ""),
            "Issue Type":       header.get("Issue Type",       ""),
            "Counter Previous": header.get("Counter Previous", ""),
            "Counter Current":  header.get("Counter Current",  ""),
            "Agenda":           issue.get("Agenda",      ""),
            "Agenda Link":      issue.get("Agenda Link", ""),
            "Root Cause":       issue.get("Root Cause",  ""),
            "Solutions":        issue.get("Solutions",   ""),
            "Has Image":        has_image,
        })
    return rows

print("Helper functions defined ✓")
print("  extract_raw / parse_header / parse_body / build_rows")

In [ ]:
# ── Cell 4: Phase 1 — Extract raw content & save preview ─────────────────────
# Run this cell and verify mom_phase1_preview.xlsx before proceeding.
# Location: same folder as your .docx files  →  PHASE1_XLSX (set in Cell 2)

raw_extractions = {}   # filename → raw dict  (reused by Cell 5)
phase1_rows     = []

docx_files = sorted(DOCX_FOLDER.glob("*.docx"))
if not docx_files:
    print(f"⚠  No .docx files found in:\n   {DOCX_FOLDER}")
else:
    for path in docx_files:
        print(f"Extracting: {path.name}")
        try:
            raw = extract_raw(path)
            raw_extractions[path.name] = raw   # keep for Cell 5
            seq = 0
            for elem in raw["body_elements"]:
                if elem["kind"] == "paragraph" and elem["text"]:
                    seq += 1
                    phase1_rows.append({
                        "Source_File": path.name,
                        "Seq":         seq,
                        "Type":        "paragraph",
                        "Content":     elem["text"],
                    })
                elif elem["kind"] == "table":
                    for row in elem["data"]:
                        line = " | ".join(c for c in row if c)
                        if line.strip():
                            seq += 1
                            phase1_rows.append({
                                "Source_File": path.name,
                                "Seq":         seq,
                                "Type":        "table_row",
                                "Content":     line,
                            })
            n_para  = sum(1 for e in raw["body_elements"] if e["kind"] == "paragraph" and e["text"])
            n_tbl   = len(raw["tables"])
            n_link  = len(raw["hyperlinks"])
            print(f"  rows={seq}  paragraphs={n_para}  tables={n_tbl}  "
                  f"links={n_link}  image={raw['has_image']}")
        except Exception as exc:
            print(f"  ✗ ERROR: {exc}")

raw_df = pd.DataFrame(phase1_rows, columns=["Source_File", "Seq", "Type", "Content"])
print(f"\nPhase 1 total: {len(raw_extractions)} file(s), {len(raw_df)} rows")

# ── Save Phase 1 preview to Excel ─────────────────────────────────────────────
with pd.ExcelWriter(str(PHASE1_XLSX), engine="openpyxl") as writer:
    raw_df.to_excel(writer, index=False, sheet_name="Phase1 Raw")
    ws = writer.sheets["Phase1 Raw"]
    ws.freeze_panes = "A2"
    for ci, cn in enumerate(raw_df.columns, 1):
        ml = max(len(cn), raw_df[cn].astype(str).map(len).max() if len(raw_df) else 0)
        ws.column_dimensions[get_column_letter(ci)].width = min(ml + 4, 80)
    for ri in range(2, len(raw_df) + 2):   # wrap Content column
        ws.cell(row=ri, column=4).alignment = Alignment(wrap_text=True, vertical="top")

print(f"\n✓ Phase 1 preview saved:")
print(f"   {PHASE1_XLSX}")
print(f"\n→ Open the file above to verify the extraction looks correct,")
print(f"  then run Cell 5 to parse fields and build the summary table.")
print()
display(raw_df.head(30))

In [ ]:
# ── Cell 5: Phase 2+3 — Parse fields and build summary rows ──────────────────
# Only run after verifying the Phase 1 preview (Cell 4) looks correct.

if not raw_extractions:
    print("⚠  No data from Cell 4. Run Cell 4 first.")
else:
    all_rows = []
    for filename, raw in raw_extractions.items():
        try:
            header = parse_header(raw)
            issues = parse_body(raw)
            rows   = build_rows(filename, header, issues, raw["has_image"])
            all_rows.extend(rows)
            filled = sum(1 for v in header.values() if v)
            print(f"  {filename}")
            print(f"    issues={len(rows)}  header fields filled={filled}/{len(header)}")
        except Exception as exc:
            print(f"  ✗ ERROR in {filename}: {exc}")
    print(f"\nTotal summary rows: {len(all_rows)}")

In [ ]:
# ── Cell 6: Build DataFrame + display preview ────────────────────────────────
df = pd.DataFrame(all_rows, columns=COLUMNS)

print(f"Shape  : {df.shape}")
print(f"Columns: {list(df.columns)}\n")

preview = df.copy()
for col in ["Agenda", "Root Cause", "Solutions"]:
    if col in preview.columns:
        preview[col] = (
            preview[col].astype(str).str[:80]
                .str.replace("\n", " ", regex=False) + "…"
        )

display(preview)

In [ ]:
# ── Cell 7: Export to mom_table.xlsx ─────────────────────────────────────────
# Output location:  OUTPUT_XLSX  (set in Cell 2 = same folder as .docx files)
from openpyxl.styles import Font, PatternFill

WRAP_COLS = {"Agenda", "Root Cause", "Solutions"}

with pd.ExcelWriter(str(OUTPUT_XLSX), engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="MoM Summary")
    ws = writer.sheets["MoM Summary"]

    ws.freeze_panes = "A2"

    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF")
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = get_column_letter(col_idx)
        col_vals   = df[col_name].astype(str)
        max_len    = max(len(str(col_name)), col_vals.map(len).max() if len(df) else 0)
        ws.column_dimensions[col_letter].width = min(max_len + 4, 60)
        if col_name in WRAP_COLS:
            for row_idx in range(2, len(df) + 2):
                ws.cell(row=row_idx, column=col_idx).alignment = Alignment(
                    wrap_text=True, vertical="top"
                )

print(f"✓  Final table saved:")
print(f"   {OUTPUT_XLSX}")
print(f"   Rows: {len(df)}  |  Columns: {len(df.columns)}")